# SegOCR — Train Worker 0 (Kaggle)

This notebook trains worker 0 of a 5-worker parallel ensemble on Kaggle. `WORKER_ID` is hardcoded below — do not change it. Run this notebook on **one** Kaggle account; run the other four worker notebooks on different accounts in parallel.

**Before running:**
1. Settings → Accelerator: **GPU T4**.
2. Add Data (sidebar) → attach **both** `segocr-ensemble-a` and `segocr-ensemble-b`. They get mounted at `/kaggle/input/segocr-ensemble-a/` and `/kaggle/input/segocr-ensemble-b/`. The cell below symlink-merges them into one directory.
3. (Optional, for resume) Add Data → Notebook Output Files → attach your previous version's output of this notebook.
4. Click **Save Version → Save & Run All** (runs server-side, survives connection issues, gives 9 hr).

When the notebook finishes, your trained weights are at `/kaggle/working/weights/averaged_best.pth` and become a downloadable notebook output.

## 1 / Setup — clone repo + install deps

In [ ]:
import os
if not os.path.isdir('/kaggle/working/segocr'):
    !git clone https://github.com/mauryantitans/SegOCR.git /kaggle/working/segocr
%cd /kaggle/working/segocr
!git pull --quiet
!pip install -q -e .
!pip install -q -r requirements/base.txt
!pip install -q segmentation-models-pytorch

## 2 / Verify GPU

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Mem: {mem_gb:.1f} GB')
else:
    raise RuntimeError('No GPU. Settings → Accelerator → GPU T4.')

## 3 / Worker config (locked to WORKER_ID = 0)

These values are baked into this notebook. The 4 other worker notebooks have the same code but different `WORKER_ID`.

In [ ]:
WORKER_ID = 0     # hardcoded for this notebook — do not change
TRAIN_SEED = WORKER_ID + 1   # different basin per worker

import os, glob, shutil
DATASET_SLUG_A = 'segocr-ensemble-a'   # indices 0..39999
DATASET_SLUG_B = 'segocr-ensemble-b'   # indices 40000..79999

def find_dataset_worker_dir(slug, worker_id):
    '''Locate /kaggle/input/.../worker{N} for a given dataset slug.
    Tries the standard mount path first, then nested variants that
    notebook-output-as-dataset workflows can produce.'''
    patterns = [
        f'/kaggle/input/{slug}/worker{worker_id}',
        f'/kaggle/input/{slug}/{slug}/worker{worker_id}',
        f'/kaggle/input/datasets/*/{slug}/worker{worker_id}',
        f'/kaggle/input/datasets/*/{slug}/{slug}/worker{worker_id}',
    ]
    for pat in patterns:
        matches = sorted(glob.glob(pat))
        if matches:
            return matches[0]
    raise FileNotFoundError(
        f'Could not find worker{worker_id} in dataset {slug!r}. '
        f'Tried: {patterns}. Did you attach the dataset via Add Data?'
    )

DATA_DIR_A = find_dataset_worker_dir(DATASET_SLUG_A, WORKER_ID)
DATA_DIR_B = find_dataset_worker_dir(DATASET_SLUG_B, WORKER_ID)
DATA_DIR = '/kaggle/working/segocr_data_merged'
WEIGHTS_DIR = '/kaggle/working/weights'
os.makedirs(WEIGHTS_DIR, exist_ok=True)

# Symlink the two halves into one merged directory so SegOCRDataset
# sees a single 16K-sample worker slice. Symlinks are ~50 bytes
# each so the working-dir overhead is ~MB, not GB.
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
for subdir in ('images', 'semantic', 'instance', 'metadata'):
    os.makedirs(f'{DATA_DIR}/{subdir}', exist_ok=True)
    for src_base in (DATA_DIR_A, DATA_DIR_B):
        src_dir = f'{src_base}/{subdir}'
        if not os.path.isdir(src_dir):
            continue
        for fname in os.listdir(src_dir):
            src = f'{src_dir}/{fname}'
            dst = f'{DATA_DIR}/{subdir}/{fname}'
            if not os.path.exists(dst):
                os.symlink(src, dst)

n_merged = len(os.listdir(f'{DATA_DIR}/images'))
print(f'Worker {WORKER_ID} of 5 (parallel ensemble)')
print(f'Dataset A: {DATA_DIR_A}')
print(f'Dataset B: {DATA_DIR_B}')
print(f'Merged:    {DATA_DIR}  ({n_merged} images via symlink)')
print(f'Weights:   {WEIGHTS_DIR}')

## 4 / Resume from previous version's output (if attached)

If a previous run of this notebook was saved and re-attached via **Add Data → Notebook Output Files**, its checkpoints get mounted somewhere under `/kaggle/input/`. We auto-detect them and copy into `/kaggle/working/weights/` so `--resume-latest` picks them up.

If this is your first run, this cell is a no-op.

In [ ]:
import shutil, glob
prev_checkpoints = []
for path in glob.glob('/kaggle/input/*/weights/checkpoint_*.pth'):
    prev_checkpoints.append(path)
for path in glob.glob('/kaggle/input/*/weights/averaged_best.pth'):
    prev_checkpoints.append(path)

if prev_checkpoints:
    print(f'Found {len(prev_checkpoints)} previous checkpoint(s); copying for resume...')
    for src in prev_checkpoints:
        dst = os.path.join(WEIGHTS_DIR, os.path.basename(src))
        shutil.copy(src, dst)
        print(f'  {src} -> {dst}')
else:
    print('No previous checkpoints attached; starting fresh.')

## 5 / Build the training config

Calibrated for Kaggle T4 + 9 hr session.

In [ ]:
import yaml
from pathlib import Path

IMG_SIZE = 512
TOTAL_ITERS = 30_000
BATCH_SIZE = 16

cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())

# Generator: fonts + paths
cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
cfg['generator']['fonts']['cache_path'] = '/kaggle/working/font_cache.json'
cfg['generator']['fonts']['min_size'] = 40
cfg['generator']['fonts']['max_size'] = 128
cfg['generator']['image_size'] = [IMG_SIZE, IMG_SIZE]
cfg['generator']['num_workers'] = 2
cfg['generator']['output_dir'] = DATA_DIR   # read from attached dataset

# Generator: text + layout + bg + composite + degradation tweaks
cfg['generator']['text']['min_length'] = 2
cfg['generator']['text']['max_length'] = 20
cfg['generator']['text']['min_words_per_line'] = 1
cfg['generator']['text']['max_words_per_line'] = 3
cfg['generator']['text']['max_lines'] = 1
cfg['generator']['text']['case_distribution'] = {
    'lower': 0.50, 'upper': 0.20, 'mixed': 0.20, 'title': 0.10,
}
cfg['generator']['text']['rare_char_boost'] = 4.0
cfg['generator']['text']['corpus_paths'] = [
    {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.30},
    {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
    {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
    {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.30},
]
cfg['generator']['layout']['modes'] = {
    'horizontal': 0.50, 'rotated': 0.20, 'curved': 0.10,
    'perspective': 0.10, 'deformed': 0.10, 'paragraph': 0.0,
}
cfg['generator']['background']['natural_image_dirs'] = []
cfg['generator']['background']['tier_distribution'] = {
    'tier1_solid': 0.40, 'tier2_procedural': 0.30,
    'tier3_natural': 0.25, 'tier4_adversarial': 0.05,
}
cfg['generator']['compositing']['color_strategy'] = {
    'contrast_aware': 0.60, 'random': 0.30, 'low_contrast': 0.10,
}
cfg['generator']['degradation']['blur'] = {
    'probability': 0.30, 'gaussian_sigma': [0.3, 1.0],
    'motion_kernel': [3, 7], 'defocus_radius': [1, 3],
}
cfg['generator']['degradation']['noise']['probability'] = 0.40
cfg['generator']['degradation']['noise']['gaussian_sigma'] = [5, 20]
cfg['generator']['degradation']['occlusion']['probability'] = 0.05

# Model
cfg['model']['architecture'] = 'unet'
cfg['model']['encoder'] = 'resnet18'
cfg['model']['encoder_weights'] = 'imagenet'
cfg['model']['head_features'] = 64
cfg['model']['decoder_channels'] = [256, 128, 64, 32, 32]
cfg['model']['heads'] = {'semantic': True, 'affinity': True, 'direction': True}
cfg['model']['num_classes'] = 63
cfg['model']['input_size'] = [IMG_SIZE, IMG_SIZE]

# Training
cfg['training']['learning_rate'] = 3e-4
cfg['training']['weight_decay'] = 1e-4
cfg['training']['total_iters'] = TOTAL_ITERS
cfg['training']['warmup_iters'] = 1_000
cfg['training']['eval_interval'] = 2_500
cfg['training']['save_interval'] = 2_500
cfg['training']['log_interval'] = 100
cfg['training']['batch_size'] = BATCH_SIZE
cfg['training']['num_workers'] = 2
cfg['training']['mixed_precision'] = True
cfg['training']['output_dir'] = WEIGHTS_DIR
cfg['training']['ema'] = {'enabled': True, 'decay': 0.999}
cfg['training']['checkpoint_averaging'] = {'enabled': True, 'top_n': 3}
cfg['training']['keep_best_n'] = 5
cfg['training']['wandb'] = {'project': None}

config_path = '/kaggle/working/train_config.yaml'
Path(config_path).write_text(yaml.safe_dump(cfg))
n_images = len(os.listdir(DATA_DIR + '/images'))
print(f'Config: {n_images} samples × {TOTAL_ITERS} iters @ {IMG_SIZE}px, batch {BATCH_SIZE}')

## 6 / Train

Training is the long step (~8 hr on T4). With **Save & Run All**, this runs server-side — close the tab if you want, your session continues. Checkpoints are written to `/kaggle/working/weights/` every 2,500 iters and the final top-3-averaged best is written there at the end.

In [ ]:
!python -m scripts.train_model --config {config_path} --resume-latest {WEIGHTS_DIR} --seed {TRAIN_SEED}

## 7 / Output files (download these)

After training completes, the cell below shows what's in `/kaggle/working/weights/`. The key file is `averaged_best.pth` — that's the final trained model you'll combine with the other 4 workers' outputs.

**To download:**

1. **Easiest:** From this notebook's page, click the **Output** tab at top, then click the download icon next to `weights/averaged_best.pth`.
2. **Via Kaggle CLI** (on your laptop, after `pip install kaggle` and setting up `~/.kaggle/kaggle.json`):
   ```bash
   kaggle kernels output YOUR_USERNAME/YOUR_NOTEBOOK_SLUG -p ./downloads/worker0
   ```

After all 5 workers are downloaded, run `scripts/average_runs.py` locally to build the ensemble.

In [ ]:
import os
print('Contents of /kaggle/working/weights/:')
for name in sorted(os.listdir(WEIGHTS_DIR)):
    size_mb = os.path.getsize(os.path.join(WEIGHTS_DIR, name)) / 1e6
    print(f'  {name:40s}  {size_mb:8.1f} MB')

print()
print('Next step: download averaged_best.pth from the Output tab')
print('        : then run scripts/average_runs.py locally after collecting all 5 workers.')